# exp080 Colab: 1-stage Combined Training (b0)

**Colab G4 (96GB VRAM) 想定**

Differences from Kaggle:
  - Drive mount (input/output)
  - Kaggle API DL for all datasets (local SSD)
  - **Batch 256** (96GB VRAM 余裕、step 数 1/8)
  - **num_workers=16**
  - **LR 8.5e-4** (sqrt(256/32) scale from base 3e-4)
  - 20 epoch (12.5h budget 内、~3-4h 完走見込)


In [1]:
# Mount Drive
from google.colab import drive
drive.mount("/content/drive")

# pip install (Colab has most preinstalled)
!pip install -q librosa timm onnxruntime

import sys, os
print(f"Python: {sys.version[:50]}")


Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 156.9 MB/s eta 0:00:00
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [2]:
# Setup Kaggle API + DL datasets
import os, json, shutil, zipfile
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/kaggle/birdclef2026")
LOCAL_DATA = Path("/content/data")
LOCAL_OUT = Path("/content/output")
DRIVE_OUT = DRIVE_ROOT / "output" / "exp080"
LOCAL_DATA.mkdir(exist_ok=True, parents=True)
LOCAL_OUT.mkdir(exist_ok=True, parents=True)
DRIVE_OUT.mkdir(exist_ok=True, parents=True)

# Kaggle credentials
KAGGLE_JSON_DRIVE = DRIVE_ROOT / "kaggle.json"
if KAGGLE_JSON_DRIVE.exists():
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    shutil.copy(str(KAGGLE_JSON_DRIVE), os.path.expanduser("~/.kaggle/kaggle.json"))
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    # Also extract KGAT token if present
    with open(KAGGLE_JSON_DRIVE) as f:
        kj = json.load(f)
    if "key" in kj and kj["key"].startswith("KGAT_"):
        os.environ["KAGGLE_API_TOKEN"] = kj["key"]
    print("kaggle.json loaded from Drive")
else:
    print(f"WARN: {KAGGLE_JSON_DRIVE} not found. Manual kaggle.json upload needed.")

from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
print(f"Kaggle API authenticated")


kaggle.json loaded from Drive
Kaggle API authenticated


In [3]:
# DL datasets to /content/data
import zipfile, time
from tqdm.auto import tqdm

def dl_competition(name, dest):
    if (dest / "train.csv").exists() and (dest / "train_audio").exists():
        print(f"  {name} already cached at {dest}")
        return
    dest.mkdir(exist_ok=True, parents=True)
    print(f"  DL competition {name} ...")
    t0 = time.time()
    api.competition_download_files(name, path=str(dest), quiet=False)
    # unzip
    zips = list(dest.glob("*.zip"))
    for zp in zips:
        with zipfile.ZipFile(zp) as zf:
            zf.extractall(dest)
        zp.unlink()
    print(f"  {name}: {(time.time()-t0)/60:.1f} min")

def dl_dataset(slug, dest):
    if dest.exists() and any(dest.iterdir()):
        print(f"  {slug} already cached at {dest}")
        return
    dest.mkdir(exist_ok=True, parents=True)
    print(f"  DL dataset {slug} ...")
    t0 = time.time()
    api.dataset_download_files(slug, path=str(dest), unzip=True, quiet=False)
    print(f"  {slug}: {(time.time()-t0)/60:.1f} min")

def dl_kernel_output(slug, dest):
    if dest.exists() and any(dest.iterdir()):
        print(f"  {slug} already cached at {dest}")
        return
    dest.mkdir(exist_ok=True, parents=True)
    print(f"  DL kernel output {slug} ...")
    t0 = time.time()
    try:
        api.kernels_output(slug, path=str(dest), force=True, quiet=False)
    except UnicodeEncodeError:
        pass
    print(f"  {slug}: {(time.time()-t0)/60:.1f} min")

# 1. BC2026 competition
dl_competition("birdclef-2026", LOCAL_DATA / "birdclef-2026")

# 2. Tucker SED ONNX
dl_dataset("tuckerarrants/bc2026-distilled-sed-public", LOCAL_DATA / "tucker_sed")

# 3. Babych BC25 1st place
dl_dataset("nikitababich/birdclef2025-1st-place-ensemble", LOCAL_DATA / "babych_b25")

# 4. XC Part 3 (dataset)
dl_dataset("maekeso/birdclef2026-xc-api-dl-part3", LOCAL_DATA / "xc_part3")

# 5. exp080a (adaptive blend pseudo, kernel output)
dl_kernel_output("maekeso/birdclef2026-exp080a-adaptive-blend", LOCAL_DATA / "exp080a")

# 6. exp080b (XC pseudo, kernel output)
dl_kernel_output("maekeso/birdclef2026-exp080b-xc-pseudo-tucker", LOCAL_DATA / "exp080b")

# 7. XC Part 1 (kernel output)
dl_kernel_output("maekeso/birdclef2026-exp047-xc-api-dl-part1", LOCAL_DATA / "xc_part1")

# 8. XC Part 2 (kernel output)
dl_kernel_output("maekeso/birdclef2026-exp047-xc-api-dl-part2", LOCAL_DATA / "xc_part2")

print(f"\nAll downloads done at {LOCAL_DATA}")


  DL competition birdclef-2026 ...


100%|██████████| 15.0G/15.0G [06:15<00:00, 42.8MB/s]



  birdclef-2026: 7.4 min
  DL dataset tuckerarrants/bc2026-distilled-sed-public ...
Dataset URL: https://www.kaggle.com/datasets/tuckerarrants/bc2026-distilled-sed-public


100%|██████████| 86.8M/86.8M [00:02<00:00, 33.4MB/s]



  tuckerarrants/bc2026-distilled-sed-public: 0.1 min
  DL dataset nikitababich/birdclef2025-1st-place-ensemble ...
Dataset URL: https://www.kaggle.com/datasets/nikitababich/birdclef2025-1st-place-ensemble


100%|██████████| 436M/436M [00:12<00:00, 36.2MB/s]



  nikitababich/birdclef2025-1st-place-ensemble: 0.3 min
  DL dataset maekeso/birdclef2026-xc-api-dl-part3 ...
Dataset URL: https://www.kaggle.com/datasets/maekeso/birdclef2026-xc-api-dl-part3


100%|██████████| 263M/263M [00:07<00:00, 35.4MB/s]



  maekeso/birdclef2026-xc-api-dl-part3: 0.2 min
  DL kernel output maekeso/birdclef2026-exp080a-adaptive-blend ...
Output file downloaded to /content/data/exp080a/adaptive_meta.json
Output file downloaded to /content/data/exp080a/file_index.json
Output file downloaded to /content/data/exp080a/primary_labels.json
Output file downloaded to /content/data/exp080a/pseudo_adaptive_234.npz
Kernel log downloaded to /content/data/exp080a/birdclef2026-exp080a-adaptive-blend.log 
  maekeso/birdclef2026-exp080a-adaptive-blend: 0.1 min
  DL kernel output maekeso/birdclef2026-exp080b-xc-pseudo-tucker ...
Output file downloaded to /content/data/exp080b/primary_labels.json
Output file downloaded to /content/data/exp080b/xc_meta.json
Output file downloaded to /content/data/exp080b/xc_pseudo_tucker.npz
Kernel log downloaded to /content/data/exp080b/birdclef2026-exp080b-xc-pseudo-tucker.log 
  maekeso/birdclef2026-exp080b-xc-pseudo-tucker: 0.1 min
  DL kernel output maekeso/birdclef2026-exp047-xc-api-dl

In [4]:
# Imports
import os, time, json, gc, math, random, re
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
import librosa
import timm
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torch.amp import GradScaler, autocast
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import tqdm.auto as tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}, torch {torch.__version__}, timm {timm.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
START = time.time()


Device: cuda, torch 2.10.0+cu128, timm 1.0.26
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition, VRAM: 102.0 GB


In [5]:
# CFG
SR = 32_000
WINDOW_SEC = 5
WINDOW_SAMPLES = SR * WINDOW_SEC

# Mel (Tucker spec)
N_MELS = 256
N_FFT = 2048
HOP_LENGTH = 512
F_MIN = 20
F_MAX = 16000
TOP_DB = 80

# Train (Colab G4, 96GB VRAM) — v2: trivial collapse 対策
N_EPOCHS = 25
BATCH_SIZE = 256       # 96GB VRAM 余裕
LR = 1.4e-3            # ★ 8.5e-4→1.4e-3: Babych base 5e-4 × sqrt(256/32) = trivial 解 脱出強化
LR_MIN = 1e-6
WEIGHT_DECAY = 1e-4
WARMUP_EPOCHS = 2      # ★ 4→2: warmup 短縮、早く target LR
DROP_PATH = 0.10       # ★ 0.15→0.10: 初期 capacity 確保
LABEL_SMOOTHING = 0.0  # ★ 0.05→0.0: hard signal sharpen

NUM_WORKERS = 16
PERSISTENT_WORKERS = True

# Aug
MIXUP_ALPHA = 0.4
MIXUP_P = 0.3     # ★ 0.5→0.3: mixup 弱める、direct learning 強化
SPECAUG_FREQ = 10
SPECAUG_TIME = 10
BG_MIX_P = 0.7    # ★ 0.5→0.7: domain shift (focal→soundscape) 攻撃強化 (Tier 1)

# Val
VAL_FRACTION = 0.20
N_CLASSES = 234
LOG_STEP_INTERVAL = 100

SEED = 42

# Paths (Colab local SSD)
DATA_PATH = LOCAL_DATA / "birdclef-2026"
TRAIN_CSV = DATA_PATH / "train.csv"
TRAIN_AUDIO_DIR = DATA_PATH / "train_audio"
TRAIN_SC_DIR = DATA_PATH / "train_soundscapes"
TAXONOMY_CSV = DATA_PATH / "taxonomy.csv"
SAMPLE_SUB = DATA_PATH / "sample_submission.csv"

# Babych b0 ckpt
BABYCH_DIR = LOCAL_DATA / "babych_b25"
BABYCH_B0_CKPT = None
for f in BABYCH_DIR.rglob("tf_efficientnet_b0*.pt"):
    BABYCH_B0_CKPT = f; break
assert BABYCH_B0_CKPT is not None, f"Babych b0 ckpt not found in {BABYCH_DIR}"
print(f"Babych b0: {BABYCH_B0_CKPT}")

# Pseudo paths
SOUNDSCAPE_PSEUDO_DIR = LOCAL_DATA / "exp080a"
XC_PSEUDO_DIR = LOCAL_DATA / "exp080b"

# XC audio paths
XC_PART1 = LOCAL_DATA / "xc_part1"
XC_PART2 = LOCAL_DATA / "xc_part2"
XC_PART3 = LOCAL_DATA / "xc_part3"

OUT_DIR = LOCAL_OUT
DRIVE_OUT_DIR = DRIVE_OUT

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"\nDATA_PATH: {DATA_PATH}")
print(f"OUT_DIR: {OUT_DIR}")
print(f"DRIVE_OUT: {DRIVE_OUT_DIR}")


Babych b0: /content/data/babych_b25/tf_efficientnet_b0.ns_jft_in1k_incest_amphibia_128_bs_0.0_drop_path_rate_20_duration_sed_type_0.5_mixup_p_(224, 512)_size_ce_4096_n_fft_full_data_22_seed_40_epoch.pt

DATA_PATH: /content/data/birdclef-2026
OUT_DIR: /content/output
DRIVE_OUT: /content/drive/MyDrive/kaggle/birdclef2026/output/exp080


In [6]:
# Load taxonomy + sample_submission
sample_sub = pd.read_csv(SAMPLE_SUB)
PRIMARY_LABELS = sample_sub.columns[1:].tolist()
assert len(PRIMARY_LABELS) == N_CLASSES
LABEL2IDX = {label: i for i, label in enumerate(PRIMARY_LABELS)}
print(f"234 species: {len(PRIMARY_LABELS)}")

taxo = pd.read_csv(TAXONOMY_CSV)
label_to_taxon = dict(zip(taxo["primary_label"].astype(str), taxo["class_name"].astype(str)))
TAXON_MASKS = {
    t: np.array([i for i, lbl in enumerate(PRIMARY_LABELS) if label_to_taxon.get(lbl, "") == t])
    for t in ["Aves", "Amphibia", "Insecta", "Mammalia", "Reptilia"]
}
print(f"Taxon: " + " ".join(f"{t}={len(m)}" for t, m in TAXON_MASKS.items()))

# Load train.csv (focal)
train_df = pd.read_csv(TRAIN_CSV)
train_df["primary_label"] = train_df["primary_label"].astype(str)
def _exists(fn): return (TRAIN_AUDIO_DIR / fn).exists()
train_df["exists"] = train_df["filename"].map(_exists)
train_df = train_df[train_df["exists"]].drop(columns=["exists"]).reset_index(drop=True)
print(f"train_df (focal, existing): {len(train_df)}")

# Stratified 80/20 split
from collections import Counter
label_counts = Counter(train_df["primary_label"])
single_label = [lbl for lbl, c in label_counts.items() if c < 2]
multi_label_df = train_df[~train_df["primary_label"].isin(single_label)].reset_index(drop=True)
single_label_df = train_df[train_df["primary_label"].isin(single_label)].reset_index(drop=True)

train_multi, val_multi = train_test_split(
    multi_label_df, test_size=VAL_FRACTION,
    stratify=multi_label_df["primary_label"], random_state=SEED,
)
train_focal_df = pd.concat([train_multi, single_label_df], ignore_index=True).reset_index(drop=True)
val_focal_df = val_multi.reset_index(drop=True)
print(f"  train_focal: {len(train_focal_df)}, val_focal: {len(val_focal_df)}")
print(f"  train sp: {train_focal_df['primary_label'].nunique()}, val sp: {val_focal_df['primary_label'].nunique()}")


234 species: 234
Taxon: Aves=162 Amphibia=35 Insecta=28 Mammalia=8 Reptilia=1
train_df (focal, existing): 35549
  train_focal: 28440, val_focal: 7109
  train sp: 206, val sp: 199


In [7]:
# Load soundscape pseudo (exp080a)
ss_npz_path = next(SOUNDSCAPE_PSEUDO_DIR.rglob("pseudo_adaptive_234.npz"))
ss_npz = np.load(ss_npz_path, allow_pickle=True)
ss_probs = ss_npz["probs"].astype(np.float32)
ss_file_ids = ss_npz["file_ids"]
print(f"SS pseudo: {ss_probs.shape}, mean={ss_probs.mean():.5f}")

ss_id_to_path = {str(fid): TRAIN_SC_DIR / f"{fid}.ogg" for fid in ss_file_ids}
print(f"  SS exists: {sum(1 for p in ss_id_to_path.values() if p.exists())} / {len(ss_file_ids)}")

# Load XC pseudo (exp080b)
xc_npz_path = next(XC_PSEUDO_DIR.rglob("xc_pseudo_tucker.npz"))
xc_npz = np.load(xc_npz_path, allow_pickle=True)
xc_probs = xc_npz["probs"].astype(np.float32)
xc_file_ids = xc_npz["file_ids"]
xc_n_actual = xc_npz["n_actual_chunks"]
print(f"\nXC pseudo: {xc_probs.shape}, mean={xc_probs.mean():.5f}, total valid chunks={int(xc_n_actual.sum())}")

# Build XC id → path map (rglob across 3 parts, tqdm 進捗)
print(f"  Building XC audio path index (rglob across Part 1+2+3)...")
xc_id_to_path_full = {}
for base in [XC_PART1, XC_PART2, XC_PART3]:
    if not base.exists():
        print(f"    {base.name}: skip (not mounted)")
        continue
    # First: count total mp3 for tqdm
    mp3_iter = base.rglob('*.mp3')
    base_files = list(tqdm.tqdm(mp3_iter, desc=f'  rglob {base.name}', unit='file'))
    print(f"    {base.name}: {len(base_files)} mp3 found")
    for fp in tqdm.tqdm(base_files, desc=f'  index {base.name}', unit='file'):
        xc_id_to_path_full[fp.stem] = fp
print(f"  Total XC mp3 indexed: {len(xc_id_to_path_full)}")

# Map xc_file_ids → path (subset of full)
xc_id_to_path = {}
for xc_id in tqdm.tqdm(xc_file_ids, desc='  match xc_id→path', unit='id'):
    s = str(xc_id)
    if s in xc_id_to_path_full:
        xc_id_to_path[s] = xc_id_to_path_full[s]
print(f"  XC matched: {len(xc_id_to_path)} / {len(xc_file_ids)}")


SS pseudo: (10658, 12, 234), mean=0.00787
  SS exists: 10658 / 10658

XC pseudo: (26488, 12, 234), mean=0.00567, total valid chunks=164044
  Building XC audio path index (rglob across Part 1+2+3)...


  rglob xc_part1: 0file [00:00, ?file/s]

    xc_part1: 495 mp3 found


  index xc_part1:   0%|          | 0/495 [00:00<?, ?file/s]

  rglob xc_part2: 0file [00:00, ?file/s]

...

    xc_part3: 111 mp3 found


  index xc_part3:   0%|          | 0/111 [00:00<?, ?file/s]

  Total XC mp3 indexed: 1101


  match xc_id→path:   0%|          | 0/26488 [00:00<?, ?id/s]

  XC matched: 1101 / 26488


In [8]:
# Model (Babych SED b0)
def gem_freq(x, p=3, eps=1e-6):
    return F.avg_pool2d(x.clamp(min=eps).pow(p), (x.size(-2), 1)).pow(1.0 / p)


class GeMFreq(nn.Module):
    def __init__(self, p=3, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps
    def forward(self, x):
        return gem_freq(x, p=self.p, eps=self.eps)


class AttHead(nn.Module):
    def __init__(self, in_chans, p=0.5, num_class=234, hidden_dim=512):
        super().__init__()
        self.pooling = GeMFreq()
        self.dense_layers = nn.Sequential(
            nn.Dropout(p / 2), nn.Linear(in_chans, hidden_dim), nn.ReLU(), nn.Dropout(p),
        )
        self.fix_scale = nn.Conv1d(hidden_dim, num_class, kernel_size=1, bias=True)

    def forward(self, feat):
        feat = self.pooling(feat).squeeze(-2).permute(0, 2, 1)
        feat = self.dense_layers(feat).permute(0, 2, 1)
        return {"framewise_logit": self.fix_scale(feat)}


class NormalizeMelSpec(nn.Module):
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps
    def forward(self, X):
        mean = X.mean((1, 2), keepdim=True)
        std = X.std((1, 2), keepdim=True)
        Xstd = (X - mean) / (std + self.eps)
        norm_max = torch.amax(Xstd, dim=(1, 2), keepdim=True)
        norm_min = torch.amin(Xstd, dim=(1, 2), keepdim=True)
        return (Xstd - norm_min) / (norm_max - norm_min + self.eps)


class SpecFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            T.MelSpectrogram(sample_rate=SR, normalized=True, n_fft=N_FFT,
                             hop_length=HOP_LENGTH, win_length=N_FFT,
                             f_max=F_MAX, n_mels=N_MELS, f_min=F_MIN),
            T.AmplitudeToDB(top_db=TOP_DB),
        )
        self.norm = NormalizeMelSpec()
    def forward(self, x):
        return self.norm(self.feature_extractor(x))


class CLEFClassifierSED(nn.Module):
    def __init__(self, num_classes=N_CLASSES, drop_path_rate=DROP_PATH):
        super().__init__()
        self.mel_spectr_generator = SpecFeatureExtractor()
        self.backbone = timm.create_model(
            "tf_efficientnet_b0.ns_jft_in1k", pretrained=True, features_only=True,
            in_chans=3, drop_path_rate=drop_path_rate,
        )
        backbone_dim = self.backbone.feature_info.channels()[-1]
        self.head = AttHead(in_chans=backbone_dim, num_class=num_classes)
        self.num_classes = num_classes

    def forward(self, wav, return_framewise=False):
        spec = self.mel_spectr_generator(wav)
        spec3 = torch.stack([spec, spec, spec], 1)
        feat = self.backbone(spec3)[-1]
        head_output = self.head(feat)
        framewise_logit = head_output["framewise_logit"]
        clip_logit = framewise_logit.max(dim=-1).values
        if return_framewise:
            return clip_logit, framewise_logit
        return clip_logit


def make_model_with_babych_init():
    model = CLEFClassifierSED()
    babych_state = torch.load(str(BABYCH_B0_CKPT), weights_only=True, map_location="cpu")
    backbone_state = {k: v for k, v in babych_state.items() if k.startswith("backbone.")}
    msg = model.load_state_dict(backbone_state, strict=False)
    print(f"  Babych load: missing={len(msg.missing_keys)}, unexpected={len(msg.unexpected_keys)}")
    return model


_tmp = make_model_with_babych_init()
print(f"Model: {sum(p.numel() for p in _tmp.parameters())/1e6:.1f}M params")
del _tmp; gc.collect()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

  Babych load: missing=7, unexpected=0
Model: 3.9M params


73

In [9]:
# Datasets — same as Kaggle version
class FocalHardDS(Dataset):
    def __init__(self, df, train_audio_dir, label2idx, ss_paths_for_bg=None, train_mode=True):
        self.df = df.reset_index(drop=True)
        self.dir = Path(train_audio_dir)
        self.label2idx = label2idx
        self.train_mode = train_mode
        self.ss_paths = ss_paths_for_bg if (train_mode and ss_paths_for_bg) else None

    def __len__(self): return len(self.df)

    def load_audio(self, filename):
        try:
            y, _ = librosa.load(str(self.dir / filename), sr=SR, mono=True)
            return y.astype(np.float32)
        except Exception:
            return np.zeros(SR * 5, dtype=np.float32)

    def crop_5s(self, y):
        if len(y) < WINDOW_SAMPLES:
            pad = WINDOW_SAMPLES - len(y)
            left = np.random.randint(0, pad + 1) if self.train_mode else pad // 2
            y = np.pad(y, (left, pad - left))
        elif len(y) > WINDOW_SAMPLES:
            start = np.random.randint(0, len(y) - WINDOW_SAMPLES + 1) if self.train_mode else (len(y) - WINDOW_SAMPLES) // 2
            y = y[start: start + WINDOW_SAMPLES]
        return y

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        y = self.load_audio(row["filename"])
        y = self.crop_5s(y)
        if self.train_mode and self.ss_paths and np.random.random() < BG_MIX_P:
            bg_path = self.ss_paths[np.random.randint(len(self.ss_paths))]
            try:
                bg, _ = librosa.load(str(bg_path), sr=SR, mono=True)
                bg = self.crop_5s(bg.astype(np.float32))
                y = 0.7 * y + 0.3 * bg
            except Exception:
                pass
        m = np.abs(y).max()
        if m > 0: y = y / m
        label = np.zeros(N_CLASSES, dtype=np.float32)
        if row["primary_label"] in self.label2idx:
            label[self.label2idx[row["primary_label"]]] = 1.0
        sec = str(row.get("secondary_labels", "")).strip()
        if sec and sec != "[]" and sec != "nan":
            for s in sec.replace("[", "").replace("]", "").replace("'", "").split(","):
                s = s.strip()
                if s in self.label2idx:
                    label[self.label2idx[s]] = 1.0
        if LABEL_SMOOTHING > 0:
            label = label * (1 - LABEL_SMOOTHING) + LABEL_SMOOTHING / N_CLASSES
        return torch.from_numpy(y), torch.from_numpy(label)


class SoundscapePseudoDS(Dataset):
    def __init__(self, file_ids, id_to_path, pseudo_array):
        self.entries = []
        for i, fid in enumerate(file_ids):
            p = id_to_path.get(str(fid))
            if p and p.exists():
                self.entries.append((str(fid), p, i))
        self.pseudo = pseudo_array

    def __len__(self): return len(self.entries)

    def __getitem__(self, idx):
        fid, path, pseudo_idx = self.entries[idx]
        win = np.random.randint(12)
        try:
            y, _ = librosa.load(str(path), sr=SR, mono=True, offset=win * 5.0, duration=5.0)
        except Exception:
            y = np.zeros(WINDOW_SAMPLES, dtype=np.float32)
        if len(y) < WINDOW_SAMPLES:
            y = np.pad(y, (0, WINDOW_SAMPLES - len(y)))
        else:
            y = y[:WINDOW_SAMPLES]
        m = np.abs(y).max()
        if m > 0: y = y / m
        target = self.pseudo[pseudo_idx, win].astype(np.float32)
        return torch.from_numpy(y.astype(np.float32)), torch.from_numpy(target)


class XCPseudoDS(Dataset):
    """XC audio (MP3): full load + pad 60s + reshape for alignment."""
    def __init__(self, file_ids, id_to_path, pseudo_array, n_actual_chunks):
        self.entries = []
        for i, fid in enumerate(file_ids):
            p = id_to_path.get(str(fid))
            n_act = int(n_actual_chunks[i])
            if p and p.exists() and n_act >= 1:
                self.entries.append((str(fid), p, i, n_act))
        self.pseudo = pseudo_array
        self.target_60s_samples = SR * 60

    def __len__(self): return len(self.entries)

    def __getitem__(self, idx):
        fid, path, pseudo_idx, n_act = self.entries[idx]
        try:
            y, _ = librosa.load(str(path), sr=SR, mono=True)
            y = y.astype(np.float32)
            if len(y) < self.target_60s_samples:
                y = np.pad(y, (0, self.target_60s_samples - len(y)))
            else:
                y = y[:self.target_60s_samples]
        except Exception:
            y = np.zeros(self.target_60s_samples, dtype=np.float32)
        chunks = y.reshape(12, WINDOW_SAMPLES)
        win = np.random.randint(n_act)
        chunk = chunks[win]
        m = np.abs(chunk).max()
        if m > 0: chunk = chunk / m
        target = self.pseudo[pseudo_idx, win].astype(np.float32)
        return torch.from_numpy(chunk.astype(np.float32)), torch.from_numpy(target)


ss_path_list = [p for p in ss_id_to_path.values() if p.exists()][:200]
train_focal_ds = FocalHardDS(train_focal_df, TRAIN_AUDIO_DIR, LABEL2IDX,
                              ss_paths_for_bg=ss_path_list, train_mode=True)
val_focal_ds = FocalHardDS(val_focal_df, TRAIN_AUDIO_DIR, LABEL2IDX, train_mode=False)
ss_ds = SoundscapePseudoDS(ss_file_ids, ss_id_to_path, ss_probs)
xc_ds = XCPseudoDS(xc_file_ids, xc_id_to_path, xc_probs, xc_n_actual)

print(f"train_focal_ds: {len(train_focal_ds)}, val_focal_ds: {len(val_focal_ds)}")
print(f"ss_ds: {len(ss_ds)}, xc_ds: {len(xc_ds)}")
train_combined_ds = ConcatDataset([train_focal_ds, ss_ds, xc_ds])
print(f"Combined: {len(train_combined_ds)}")


train_focal_ds: 28440, val_focal_ds: 7109
ss_ds: 10658, xc_ds: 1101
Combined: 40199


In [10]:
# Helpers
def mixup_audio(wav, label, alpha=MIXUP_ALPHA, p=MIXUP_P):
    if np.random.random() >= p:
        return wav, label
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(wav.size(0), device=wav.device)
    return lam * wav + (1 - lam) * wav[idx], lam * label + (1 - lam) * label[idx]


@torch.no_grad()
def eval_val_focal_macro(model, val_dl, device):
    model.eval()
    all_preds, all_labels = [], []
    for wav, label in val_dl:
        wav = wav.to(device, non_blocking=True)
        with autocast('cuda'):
            clip_logit = model(wav)
        all_preds.append(torch.sigmoid(clip_logit).float().cpu().numpy())
        all_labels.append(label.numpy())
    preds = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)
    labels_bin = (labels > 0.5).astype(np.float32)
    aucs = []
    for sp in range(N_CLASSES):
        if labels_bin[:, sp].sum() > 0 and labels_bin[:, sp].sum() < len(labels_bin):
            try:
                aucs.append(roc_auc_score(labels_bin[:, sp], preds[:, sp]))
            except Exception:
                pass
    return float(np.mean(aucs)) if aucs else float("nan"), len(aucs)


def class_stats_str(aucs):
    if not aucs:
        return "n=0"
    vals = np.array(aucs)
    return (f"n={len(vals)} median={np.median(vals):.3f} p25={np.percentile(vals,25):.3f} "
            f"p75={np.percentile(vals,75):.3f} #>0.5={int((vals>0.5).sum())} #>0.7={int((vals>0.7).sum())} "
            f"#>0.9={int((vals>0.9).sum())} #perfect={int((vals>=1.0).sum())}")


@torch.no_grad()
def eval_val_full(model, val_dl, device):
    """Returns val_focal_macro + per-taxon + class_stats (M7-style)."""
    model.eval()
    all_preds, all_labels = [], []
    for wav, label in val_dl:
        wav = wav.to(device, non_blocking=True)
        with autocast('cuda'):
            clip_logit = model(wav)
        all_preds.append(torch.sigmoid(clip_logit).float().cpu().numpy())
        all_labels.append(label.numpy())
    preds = np.concatenate(all_preds)
    labels_bin = (np.concatenate(all_labels) > 0.5).astype(np.float32)
    # per-class AUC
    per_sp_aucs = {}
    for sp in range(N_CLASSES):
        if labels_bin[:, sp].sum() > 0 and labels_bin[:, sp].sum() < len(labels_bin):
            try:
                per_sp_aucs[sp] = roc_auc_score(labels_bin[:, sp], preds[:, sp])
            except Exception: pass
    macro = float(np.mean(list(per_sp_aucs.values()))) if per_sp_aucs else float("nan")
    # Per-taxon
    taxon_aucs = {}
    for t, mask in TAXON_MASKS.items():
        tx_vals = [per_sp_aucs[sp] for sp in mask if sp in per_sp_aucs]
        taxon_aucs[t] = float(np.mean(tx_vals)) if tx_vals else float("nan")
    return macro, len(per_sp_aucs), taxon_aucs, list(per_sp_aucs.values())


In [11]:
# Training loop (with M7-style logging restored)
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

train_dl = DataLoader(train_combined_ds, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
                       persistent_workers=PERSISTENT_WORKERS, prefetch_factor=6)
val_dl = DataLoader(val_focal_ds, batch_size=BATCH_SIZE, shuffle=False,
                     num_workers=NUM_WORKERS, pin_memory=True,
                     persistent_workers=PERSISTENT_WORKERS)

model = make_model_with_babych_init().to(DEVICE)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
warmup_iters = WARMUP_EPOCHS * len(train_dl)
total_iters = N_EPOCHS * len(train_dl)
sched_warmup = LinearLR(optimizer, start_factor=1/25, end_factor=1.0, total_iters=warmup_iters)
sched_cosine = CosineAnnealingLR(optimizer, T_max=total_iters - warmup_iters, eta_min=LR_MIN)
scheduler = SequentialLR(optimizer, schedulers=[sched_warmup, sched_cosine], milestones=[warmup_iters])
scaler = GradScaler('cuda')

best_val = -1.0
history = []
total_steps = len(train_dl)
print(f"=== Training: {N_EPOCHS} epochs × {total_steps} steps, batch={BATCH_SIZE} ===")

for epoch in range(N_EPOCHS):
    t0_ep = time.time()
    model.train()
    tr_loss_sum, bce_sum, n_seen = 0.0, 0.0, 0
    for step, (wav, label) in enumerate(train_dl):
        wav = wav.to(DEVICE, non_blocking=True)
        label = label.to(DEVICE, non_blocking=True)
        wav, label = mixup_audio(wav, label)
        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda'):
            clip_logit, framewise_logit = model(wav, return_framewise=True)
            frame_max_logit = framewise_logit.max(dim=-1).values
            loss_clip = F.binary_cross_entropy_with_logits(clip_logit, label)
            loss_frame = F.binary_cross_entropy_with_logits(frame_max_logit, label)
            bce = 0.5 * loss_clip + 0.5 * loss_frame
            loss = bce
        if not torch.isfinite(loss):
            continue
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        tr_loss_sum += loss.item() * wav.size(0)
        bce_sum += bce.item() * wav.size(0)
        n_seen += wav.size(0)
        if step % LOG_STEP_INTERVAL == 0 or step == total_steps - 1:
            cur_lr = optimizer.param_groups[0]["lr"]
            print(f"  [ep{epoch+1} step {step}/{total_steps}] loss={loss.item():.4f} bce={bce.item():.4f} lr={cur_lr:.2e}")

    tr_loss = tr_loss_sum / max(n_seen, 1)
    bce_avg = bce_sum / max(n_seen, 1)

    # Val (full M7-style)
    val_macro, n_valid_sp, taxon_aucs, all_aucs = eval_val_full(model, val_dl, DEVICE)
    ep_time = (time.time() - t0_ep) / 60
    total_time = (time.time() - START) / 60
    cur_lr = optimizer.param_groups[0]["lr"]
    is_best = val_macro > best_val
    best_tag = "BEST " if is_best else ""
    tax_line = "taxon: " + " ".join(f"{t}={taxon_aucs[t]:.3f}" if not math.isnan(taxon_aucs[t]) else f"{t}=nan"
                                     for t in ["Insecta", "Reptilia", "Amphibia", "Mammalia", "Aves"])
    cls_line = class_stats_str(all_aucs)

    print(f"=== Ep {epoch+1}/{N_EPOCHS}: loss={tr_loss:.4f} (bce={bce_avg:.4f}) "
          f"val_focal_macro={val_macro:.4f} ({n_valid_sp} sp) {best_tag}lr={cur_lr:.2e} "
          f"({ep_time:.1f}min, total {total_time:.1f}min) ===")
    print(f"    {tax_line}")
    print(f"    class: {cls_line}")

    if is_best:
        best_val = val_macro
        torch.save({"model_state": {k: v.cpu() for k, v in model.state_dict().items()},
                    "epoch": epoch, "val_focal_macro": val_macro},
                   OUT_DIR / "m_single_ckpt_best.pth")
        print(f"    BEST saved val={val_macro:.4f}")

    history.append({
        "ep": epoch, "tr_loss": tr_loss, "bce": bce_avg,
        "val_focal_macro": val_macro, "n_valid_sp": n_valid_sp,
        "taxon": taxon_aucs, "lr": cur_lr, "ep_time_min": ep_time, "best": is_best,
    })

    # Drive mirror per epoch (safety against Colab disconnect)
    try:
        shutil.copy(OUT_DIR / "m_single_ckpt_best.pth", DRIVE_OUT_DIR / "m_single_ckpt_best.pth")
        with open(DRIVE_OUT_DIR / "history.json", "w") as f:
            json.dump(history, f, indent=2, default=str)
    except Exception as e:
        print(f"  Drive mirror warn: {e}")

# Final
torch.save({"model_state": {k: v.cpu() for k, v in model.state_dict().items()},
            "epoch": N_EPOCHS - 1, "history": history},
           OUT_DIR / "m_single_ckpt_final.pth")
with open(OUT_DIR / "history.json", "w") as f:
    json.dump(history, f, indent=2, default=str)
print(f"\nTraining DONE: best val={best_val:.4f}, total {(time.time()-START)/60:.1f} min")


  Babych load: missing=7, unexpected=0
=== Training: 25 epochs × 157 steps, batch=256 ===
  [ep1 step 0/157] loss=2.0714 bce=2.0714 lr=6.03e-05
  [ep1 step 100/157] loss=0.0368 bce=0.0368 lr=4.88e-04
  [ep1 step 156/157] loss=0.0383 bce=0.0383 lr=7.28e-04
=== Ep 1/25: loss=0.2963 (bce=0.2963) val_focal_macro=0.5365 (199 sp) BEST lr=7.28e-04 (1.7min, total 1.8min) ===
    taxon: Insecta=0.573 Reptilia=nan Amphibia=0.563 Mammalia=0.503 Aves=0.533
    class: n=199 median=0.537 p25=0.474 p75=0.588 #>0.5=128 #>0.7=17 #>0.9=5 #perfect=0
    BEST saved val=0.5365
  [ep2 step 0/157] loss=0.0355 bce=0.0355 lr=7.32e-04
  [ep2 step 100/157] loss=0.0327 bce=0.0327 lr=1.16e-03
  [ep2 step 156/157] loss=0.0310 bce=0.0310 lr=1.40e-03
=== Ep 2/25: loss=0.0335 (bce=0.0335) val_focal_macro=0.5938 (199 sp) BEST lr=1.40e-03 (1.5min, total 3.3min) ===
    taxon: Insecta=0.287 Reptilia=nan Amphibia=0.735 Mammalia=0.543 Aves=0.577
    class: n=199 median=0.585 p25=0.521 p75=0.652 #>0.5=162 #>0.7=34 #>0.9=8 #

In [12]:
# ONNX export
print("=== ONNX export ===")
import shutil
best_ckpt = torch.load(OUT_DIR / "m_single_ckpt_best.pth", weights_only=False, map_location="cpu")
export_model = CLEFClassifierSED().cpu().eval()
export_model.load_state_dict(best_ckpt["model_state"])

dummy_wav = torch.randn(1, WINDOW_SAMPLES, dtype=torch.float32)
with torch.no_grad():
    out = export_model(dummy_wav)
    print(f"  Trace: {out.shape}")

onnx_path = OUT_DIR / "m_single_best.onnx"
try:
    torch.onnx.export(
        export_model, dummy_wav, str(onnx_path),
        input_names=["wav"], output_names=["clip_logits"],
        dynamic_axes={"wav": {0: "batch"}, "clip_logits": {0: "batch"}},
        opset_version=17, dynamo=False,
    )
    print(f"  ONNX exported: {onnx_path} ({onnx_path.stat().st_size/1e6:.1f} MB)")
except Exception as e:
    print(f"  ONNX export FAILED: {e}")

try:
    import onnxruntime as ort
    sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    onnx_out = sess.run(None, {"wav": dummy_wav.numpy()})[0]
    diff = np.abs(out.numpy() - onnx_out).max()
    print(f"  PyTorch vs ONNX diff: {diff:.6e}")
except Exception as e:
    print(f"  ONNX check failed: {e}")

# OpenVINO IR export (.xml + .bin, 3-5x CPU speedup vs ONNX)
print("\n=== OpenVINO IR export ===")
!pip install -q openvino
try:
    import openvino as ov
    ov_model = ov.convert_model(str(onnx_path))
    ov_xml = OUT_DIR / "m_single_best.xml"
    ov.save_model(ov_model, str(ov_xml))
    ov_bin = OUT_DIR / "m_single_best.bin"
    print(f"  OV IR: {ov_xml.name} ({ov_xml.stat().st_size/1e6:.2f} MB), {ov_bin.name} ({ov_bin.stat().st_size/1e6:.2f} MB)")
    # Speed sanity check
    try:
        core = ov.Core()
        compiled = core.compile_model(ov_model, "CPU")
        ov_out = compiled([dummy_wav.numpy()])[compiled.output(0)]
        ov_diff = np.abs(out.numpy() - ov_out).max()
        print(f"  PyTorch vs OpenVINO diff: {ov_diff:.6e}")
    except Exception as e:
        print(f"  OV runtime check failed: {e}")
except Exception as e:
    print(f"  OpenVINO export FAILED: {e}")
    import traceback; traceback.print_exc()


=== ONNX export ===


/tmp/ipykernel_672/1106391425.py:15: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


  Trace: torch.Size([1, 234])


/tmp/ipykernel_672/2204283022.py:76: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if return_framewise:
/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/jit_utils.py:305: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. (Triggered internally at /pytorch/torch/csrc/jit/passes/onnx/constant_fold.cpp:178.)
  _C._jit_pass_onnx_node_shape_type_inference(node, params_dict, opset_version)


  ONNX export FAILED: STFT does not currently support complex types  [Caused by the value '422 defined in (%422 : Float(*, *, strides=[162048, 1], requires_grad=0, device=cpu) = onnx::Reshape[allowzero=0](%411, %421), scope: __main__.CLEFClassifierSED::/__main__.SpecFeatureExtractor::mel_spectr_generator/torch.nn.modules.container.Sequential::feature_extractor/torchaudio.transforms._transforms.MelSpectrogram::feature_extractor.0/torchaudio.transforms._transforms.Spectrogram::spectrogram # /usr/local/lib/python3.12/dist-packages/torch/functional.py:680:0
)' (type 'Tensor') in the TorchScript graph. The containing node has kind 'onnx::Reshape'.] 
    (node defined in /usr/local/lib/python3.12/dist-packages/torch/functional.py(680): stft
/usr/local/lib/python3.12/dist-packages/torchaudio/functional/functional.py(123): spectrogram
/usr/local/lib/python3.12/dist-packages/torchaudio/transforms/_transforms.py(111): forward
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py(176

Traceback (most recent call last):
  File "/tmp/ipykernel_672/1106391425.py", line 39, in <cell line: 0>
    ov_model = ov.convert_model(str(onnx_path))
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/openvino/tools/ovc/convert.py", line 115, in convert_model
    ov_model, _ = _convert(cli_parser, params, True)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/openvino/tools/ovc/convert_impl.py", line 587, in _convert
    raise e
  File "/usr/local/lib/python3.12/dist-packages/openvino/tools/ovc/convert_impl.py", line 504, in _convert
    argv = pack_params_to_args_namespace(args, cli_parser, python_api_used)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/openvino/tools/ovc/convert_impl.py", line 404, in pack_params_to_args_namespace
    argv = args_to_argv(**args)
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/loc

In [13]:
# Mirror to Drive + (optional) Kaggle Dataset upload
import shutil

# Mirror to Drive (.pth + .onnx + OpenVINO .xml/.bin + history)
for fname in [
    "m_single_ckpt_best.pth", "m_single_ckpt_final.pth",
    "m_single_best.onnx",
    "m_single_best.xml", "m_single_best.bin",
    "history.json",
]:
    src = OUT_DIR / fname
    if src.exists():
        dst = DRIVE_OUT_DIR / fname
        shutil.copy(str(src), str(dst))
        print(f"  Mirror: {fname} → {dst} ({dst.stat().st_size/1e6:.1f} MB)")

# Kaggle Dataset upload (for inference NB to consume)
KAGGLE_DS_SLUG = "birdclef2026-exp080-train-output"
DS_TITLE = "BirdCLEF2026 exp080 train output"

ds_dir = OUT_DIR / "kaggle_ds"
ds_dir.mkdir(exist_ok=True)
for fname in [
    "m_single_ckpt_best.pth", "m_single_best.onnx",
    "m_single_best.xml", "m_single_best.bin",
    "history.json",
]:
    src = OUT_DIR / fname
    if src.exists(): shutil.copy(str(src), str(ds_dir / fname))

meta = {"title": DS_TITLE, "id": f"maekeso/{KAGGLE_DS_SLUG}", "licenses": [{"name": "CC0-1.0"}]}
with open(ds_dir / "dataset-metadata.json", "w") as f:
    json.dump(meta, f, indent=2)

try:
    api.dataset_create_new(folder=str(ds_dir), public=False, dir_mode="zip", quiet=False)
    print(f"  Created Kaggle Dataset: maekeso/{KAGGLE_DS_SLUG}")
except Exception:
    try:
        api.dataset_create_version(folder=str(ds_dir), version_notes="initial",
                                    dir_mode="zip", quiet=False)
        print(f"  Updated Kaggle Dataset: maekeso/{KAGGLE_DS_SLUG}")
    except Exception as e:
        print(f"  Upload failed (manual upload via Drive needed): {str(e)[:200]}")

print(f"\n=== exp080 Colab DONE ===")
print(f"Best val_focal_macro: {best_val:.4f}")
print(f"Total time: {(time.time()-START)/60:.1f} min")


  Mirror: m_single_ckpt_best.pth → /content/drive/MyDrive/kaggle/birdclef2026/output/exp080/m_single_ckpt_best.pth (16.8 MB)
  Mirror: m_single_ckpt_final.pth → /content/drive/MyDrive/kaggle/birdclef2026/output/exp080/m_single_ckpt_final.pth (16.9 MB)
  Mirror: history.json → /content/drive/MyDrive/kaggle/birdclef2026/output/exp080/history.json (0.0 MB)
Starting upload for file history.json


100%|██████████| 10.6k/10.6k [00:00<00:00, 29.4kB/s]


Upload successful: history.json (11KB)
Starting upload for file m_single_ckpt_best.pth


100%|██████████| 16.1M/16.1M [00:01<00:00, 11.1MB/s]


Upload successful: m_single_ckpt_best.pth (16MB)
  Created Kaggle Dataset: maekeso/birdclef2026-exp080-train-output

=== exp080 Colab DONE ===
Best val_focal_macro: 0.9550
Total time: 38.6 min


In [14]:
# Auto-disconnect (Colab Pro+ unit saving)
from google.colab import runtime
# Comment out if you want to inspect manually
# runtime.unassign()
print("Done. Uncomment runtime.unassign() to auto-disconnect.")


Done. Uncomment runtime.unassign() to auto-disconnect.
